# Ingestão Bronze


In [0]:
import re
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import to_timestamp, current_timestamp


# Padronização dos nomes dos schemas e volumes
CATALOG = "vcredit_medallion"
BRONZE_SCHEMA_NAME = "bronze"
SILVER_SCHEMA_NAME = "silver"
GOLD_SCHEMA_NAME = "gold"

BRONZE_SCHEMA = f"{CATALOG}.{BRONZE_SCHEMA_NAME}"
SILVER_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA_NAME}"
GOLD_SCHEMA = f"{CATALOG}.{GOLD_SCHEMA_NAME}"

LANDING_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA_NAME}/landing"

# Símbolos padronizados de log
OK = "✅"
FAIL = "❌"
WARN = "⚠️"

# Para verificação visual
print(f"CATALOG: {CATALOG}")
print(f"BRONZE_SCHEMA: {BRONZE_SCHEMA}")
print(f"SILVER_SCHEMA: {SILVER_SCHEMA}")
print(f"GOLD_SCHEMA: {GOLD_SCHEMA}")
print(f"LANDING_PATH: {LANDING_PATH}")



CATALOG: vcredit_medallion
BRONZE_SCHEMA: vcredit_medallion.bronze
SILVER_SCHEMA: vcredit_medallion.silver
GOLD_SCHEMA: vcredit_medallion.gold
LANDING_PATH: /Volumes/vcredit_medallion/bronze/landing


In [0]:
# Mapeamento explícito arquivo -> nome de tabela.
expected_files = {
    "base_atendentes.csv":      ("base_atendentes",      "UTF-8"),
    "base_motivos.csv":         ("base_motivos",         "UTF-8"),
    "canais.csv":               ("canais",               "UTF-8"),
    "chamados.csv":             ("chamados",             "UTF-8"),
    "chamados_hora.csv":        ("chamados_hora",        "ISO-8859-1"),
    "clientes.csv":             ("clientes",             "ISO-8859-1"),
    "custos.csv":               ("custos",               "UTF-8"),
    "pesquisa_satisfacao.csv":  ("pesquisa_satisfacao",  "UTF-8"),
}

for file, (name, encoding) in expected_files.items():
    table = f"{BRONZE_SCHEMA}.tb_{name}"
    file_path = f"{LANDING_PATH}/{file}"

    # Leitura: CSV bruto, sem inferência de schema.
    # Todas as colunas (não de metadados) entram como string.
    # Coluna referente a hora da ingestão, o _ antes do nome explicita ser uma coluna de metadados.
    df = spark.read.csv(file_path, header=True, encoding=encoding).withColumn(
        "_ingerido_em", current_timestamp()
    )

    # Escrita: Delta, modo overwrite (batelada única, sem ingestão incremental)
    #    e overwriteSchema para acomodar variações de tipo entre reexecuções.
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )

    # Validação: tabela vazia é erro
    count = spark.table(table).count()
    if count == 0:
        print(f"{FAIL} {table} foi gravada vazia")
        raise Exception(
            f"Tabela {table} vazia. Verifique o arquivo {file} na landing zone."
        )
    else:
        print(f"{OK} {table}: {count} linhas")

    # Inspeção visual: amostra das 5 primeiras linhas para conferência rápida.
    display(spark.table(table).limit(5))

✅ vcredit_medallion.bronze.tb_base_atendentes: 25 linhas


id_atendente,nome_atendente,nivel_atendimento,data_admissao,data_desligamento,status_atendente,_ingerido_em
1,Marcela França,1,18/04/2019,16/11/2025,inativo,2026-10-25T15:18:02.635Z
2,Zilda Esteves,1,12/09/2023,24/01/2026,inativo,2026-10-25T15:18:02.635Z
3,William Marques,1,28/05/2019,15/04/2025,inativo,2026-10-25T15:18:02.635Z
4,Solange Silveira,Nível 1,07/10/2019,null,ativo,2026-10-25T15:18:02.635Z
5,Joaquim Barros,Nível 1,06/12/2020,23/05/2026,inativo,2026-10-25T15:18:02.635Z


✅ vcredit_medallion.bronze.tb_base_motivos: 14 linhas


id_motivo,nome_motivo,categoria,criticidade,_ingerido_em
11,Bloqueio de cartão,Cartao,Alta,2026-10-25T15:18:06.324Z
7,Compra não autorizada,Cartao,alta,2026-10-25T15:18:06.324Z
13,Dúvidas gerais sobre programa de pontos,Beneficios,baixa,2026-10-25T15:18:06.324Z
14,Transferência de agência,null,Media,2026-10-25T15:18:06.324Z
9,Problema com aplicativo,Atendimento,Media,2026-10-25T15:18:06.324Z


✅ vcredit_medallion.bronze.tb_canais: 4 linhas


...

id_cliente,nome,email,regiao,idade,data_cadastro,dia_vencimento_fatura,_ingerido_em
56324416706,Sérgio Martins Miranda,sergio.miranda98@hotmail.com,Sul,51,14/10/2024,10,2026-10-25T15:18:23.108Z
97653229900,Leandro Magalhães Lopes,lopes.leandro@yahoo.com.br,Sudeste,51,2026-07-07,10,2026-10-25T15:18:23.108Z
15132904850,Giovana Soares Vasconcelos,giovana.vasconcelos26@hotmail.com,Sudeste,52,16/02/2019,25,2026-10-25T15:18:23.108Z
62566924227,Adriana Nascimento,adriana96@gmail.com,Nordeste,39,27/08/2025,20,2026-10-25T15:18:23.108Z
35213062014,Vera Duarte,vera.duarte@ig.com.br,Sudeste,34,07/05/2026,15,2026-10-25T15:18:23.108Z


✅ vcredit_medallion.bronze.tb_custos: 131069 linhas


id_custo,id_chamado,custo,_ingerido_em
114049,110377,"0,39",2026-10-25T15:18:27.110Z
87679,83693,"R$ 1,46",2026-10-25T15:18:27.110Z
92429,88501,"12,24",2026-10-25T15:18:27.110Z
58033,53673,"24,43",2026-10-25T15:18:27.110Z
31491,26832,11.14reais,2026-10-25T15:18:27.110Z


✅ vcredit_medallion.bronze.tb_pesquisa_satisfacao: 81021 linhas


id_pesquisa,id_chamado,nota_atendimento,_ingerido_em
71619,115746,3,2026-10-25T15:18:31.117Z
13534,19389,4,2026-10-25T15:18:31.117Z
53852,83856,4,2026-10-25T15:18:31.117Z
5563,7771,5,2026-10-25T15:18:31.117Z
15959,22934,5,2026-10-25T15:18:31.117Z
